In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2016.csv")

In [ ]:
df.info()

In [4]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [5]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [6]:
import pandas as pd

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    errors='coerce',     
    dayfirst=True         
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')

C:\Users\hp\AppData\Local\Temp\ipykernel_11892\3934601853.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [7]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [8]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [9]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [10]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [11]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [12]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [42]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [43]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [ ]:
print("Rows deleted:", (df_deduped))

In [21]:
len(dfc)

55275

In [44]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [24]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, is_duplicate_candidate, price_outlier, price_corrected_flag]
Index: []

[0 rows x 37 columns]


In [ ]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"   
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [26]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [27]:
dfc["payment_method"].unique()

array(['Cash on Delivery', 'Credit Card', 'Debit Card', 'Net Banking',
       'UPI'], dtype=object)

In [28]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2016_00000001', 'TXN_2016_00000002', 'TXN_2016_00000003', 'TXN_2016_00000004', 'TXN_2016_00000005', 'TXN_2016_00000006', 'TXN_2016_00000007', 'TXN_2016_00000008', 'TXN_2016_00000009', 'TXN_2016_00000010', 'TXN_2016_00000011', 'TXN_2016_00000012', 'TXN_2016_00000013', 'TXN_2016_00000014', 'TXN_2016_00000015', 'TXN_2016_00000016', 'TXN_2016_00000017', 'TXN_2016_00000018', 'TXN_2016_00000019', 'TXN_2016_00000020', 'TXN_2016_00000021', 'TXN_2016_00000022', 'TXN_2016_00000023', 'TXN_2016_00000024', 'TXN_2016_00000025', 'TXN_2016_00000026', 'TXN_2016_00000027', 'TXN_2016_00000028', 'TXN_2016_00000029', 'TXN_2016_00000030', 'TXN_2016_00000031', 'TXN_2016_00000032', 'TXN_2016_00000033', 'TXN_2016_00000034', 'TXN_2016_00000035', 'TXN_2016_00000036', 'TXN_2016_00000037', 'TXN_2016_00000038', 'TXN_2016_00000039', 'TXN_2016_00000040', 'TXN_2016_00000041', 'TXN_2016_00000042', 'TXN_2016_00000043', 'TXN_2016_00000044', 'TXN_2016_00000045', 'TXN_2

In [29]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0']

final_amount_inr: ['10000.86', '100002.66', '10002.32', '100021.70999999999', '100026.71', '10003.47', '10004.21', '10004.69', '10005.54', '10006.38', '10007.92', '100070.42', '100072.84', '100077.66', '10008.2', '100081.54', '100091.8', '100095.99', '10011.66', '10011.84', '100131.56', '100151.37', '100166.58', '10019.37', '100193.1', '100193.3', '100193.56', '100195.55', '100197.8', '100200.26', '10023.12', '10024.37', '100240.87', '100247.59', '100257.26', '100259.03', '100260.88', '100269.23', '10027.98', '100287.49', '10029.34', '100298.01', '10030.55', '10030.64', '100300.4', '100308.94', '10032.6', '10032.68', '100337.78', '100338.86', '10034.16', '10034.41', '10034.93', '100359.31', '100366.76', '100369.89', '100372.29', '10038.45', '100394.05', '100410.62', '100426.0', '10043.23', '10043.28', '100432.26', '10044.62', '100443.95999999999', '10045.7', '10047.44', '100473.26', '100473.71', '100474.73', '10049.78', '100501

In [30]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2016']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.1', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.29', '0.32', '0.33', '0.34', '0.35', '0.4', '0.42', '0.43', '0.45', '0.46', '0.47', '0.48', '0.49', '0.52', '0.53', '0.55', '0.56', '0.57', '0.58', '0.59', '0.62', '0.64', '0.65', '0.66', '0.72', '0.73', '0.75', '0.78', '1.21', '1.29', '1.39', '1.4', '1.5', '1.62', '1.74', '1.76', '1.79', '1.98', '1.99', '2.01', '2.02', '2.06', '2.17', '2.18', '2.26', '2.27', '2.33', '2.49', '2.57', '2.6', '2.66', '2.68', '2.69', '2.7', '2.75', '2.8', '21.84', '25.87', '26.43', '33.52', '44.26']

is_prime_eligible: ['False', 'True']

product_rating: ['3.0', '3

In [45]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2016_clean.csv",header='infer',index=False)

In [32]:
dfc.to_csv(
    r"amazon_final_clean_2016.csv",
    index=False,header='infer'
)

In [ ]:
dfc

In [39]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [40]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
